# SWOT L2 virtualizarr example
### (NASA CMR Shortname: SWOT_L2_LR_SSH_Basic_D)

#### *Author: Ed Armstrong, PO.DAAC*

*Reference herein to any specific commercial product, process, or service by trade name, trademark, manufacturer, or otherwise, does not constitute or imply its endorsement by the United States Government or the Jet Propulsion Laboratory, California Institute of Technology.*

In [55]:
# Built-in packages
import os
import sys
from pathlib import Path

# Filesystem management 
import fsspec
import earthaccess

# Data handling
import xarray as xr
from virtualizarr import open_virtual_dataset

# Parallel computing 
import multiprocessing
import dask
from dask import delayed
import dask.array as da
from dask.distributed import Client

# Pandas
import pandas as pd

# Numpy
import numpy as np

# Other mapping
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Coiled for Dask management
import coiled

In [56]:
 pip list | grep -E  -e  '(^xarray|virtualizarr|earthaccess|kerchunk|zarr)'

if: Expression Syntax.
then: Command not found.
earthaccess               0.14.0
kerchunk                  0.2.7
virtualizarr              1.3.2
xarray                    2025.1.1
zarr                      2.18.4
Note: you may need to restart the kernel to use updated packages.


## 1. Get Data File S3 endpoints in Earthdata Cloud 
The first step is to find the S3 endpoints to the files. Handling access credentials to Earthdata and then finding the endpoints can be done a number of ways (e.g. using the `requests`, `s3fs` packages) but we use the `earthaccess` package for its ease of use. We get the endpoints for all files in the CCMP record.

In [57]:
# Get Earthdata creds
earthaccess.login()

In [58]:
# Locate  SWOT L2 Version D file information / metadata for the PGD files.
# The start of the science orbits files is 2023-07-26 (cycle 1)
granule_info_1 = earthaccess.search_data(
    short_name="SWOT_L2_LR_SSH_Basic_D",
    granule_name="SWOT_L2_LR_SSH_Basic*PGD*.nc",
    #temporal=("2023-03-27", "2025-04-08"),
    temporal=("2023-07-26", "2025-04-08"),
    )

In [59]:
# Locate  SWOT L2 file Version D information / metadata for the PID files:
granule_info_2 = earthaccess.search_data(
    short_name="SWOT_L2_LR_SSH_Basic_D",
    granule_name="SWOT_L2_LR_SSH_Basic*PID*.nc",
    temporal=("2025-05-06", "2026-03-22"),
    )

In [60]:
# append the two granule lists. They are sorted in time.
granule_info = granule_info_1 + granule_info_2

In [61]:
# Get S3 or HTTPS endpoints for all files:
data_links = [g.data_links(access="https")[0] for g in granule_info]
#data_links = [g.data_links(access="direct")[0] for g in granule_info]
print(len(data_links))
print(data_links[0:4])

25757
['https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_D/SWOT_L2_LR_SSH_Basic_001_134_20230726T000155_20230726T000155_PGD0_02.nc', 'https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_D/SWOT_L2_LR_SSH_Basic_001_135_20230726T005326_20230726T005326_PGD0_02.nc', 'https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_D/SWOT_L2_LR_SSH_Basic_001_136_20230726T014449_20230726T014449_PGD0_02.nc', 'https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_D/SWOT_L2_LR_SSH_Basic_001_137_20230726T023620_20230726T023620_PGD0_02.nc']


In [62]:
# Get AWS creds. Note that if you spend more than 1 hour in the notebook, you may have to re-run this line!!!
fs = earthaccess.get_fsspec_https_session()
#fs = earthaccess.get_s3_filesystem(results=granule_info)

## 2. Generate and open reference files 

In [63]:
# This will be assigned to 'loadable_variables' and needs to be modified per the specific 
# coord names of the data set:
coord_vars = ["num_lines","num_pixels"]
reader_opts = {"storage_options": fs.storage_options,   "timeout": 120} # S3 filesystem creds from previous section.

The reference can be saved to file and used to open the corresponding CCMP data file with Xarray:

### 2.2.1 Process some files using Dask local cluster in parallel
If using an `m6i.4xlarge` AWS EC2 instance, there are 16 CPUs available and each should have enough memory to utilize all at once. If working on a different VM-type, change the `n_workers` in the call to `Client()` below as needed.

In [64]:
# Check how many cpu's are on this VM:
print("CPU count =", multiprocessing.cpu_count())

CPU count = 16


In [65]:
# Start up cluster and print some information about it:
client = Client(n_workers=4, threads_per_worker=1)

dask.config.set({"distributed.worker.connections.outgoing": 5})

print(client.cluster)
print("View any work being done on the cluster here", client.dashboard_link)

LocalCluster(de846da8, 'tcp://127.0.0.1:60068', workers=4, threads=4, memory=64.00 GiB)
View any work being done on the cluster here http://127.0.0.1:8787/status


2026-04-02 16:25:24,646 - distributed.scheduler - WARNING - Worker failed to heartbeat for 309s; attempting restart: <WorkerState 'tcp://127.0.0.1:60079', name: 0, status: running, memory: 0, processing: 0>
2026-04-02 16:25:24,648 - distributed.scheduler - WARNING - Worker failed to heartbeat for 309s; attempting restart: <WorkerState 'tcp://127.0.0.1:60080', name: 3, status: running, memory: 0, processing: 0>
2026-04-02 16:25:24,649 - distributed.scheduler - WARNING - Worker failed to heartbeat for 309s; attempting restart: <WorkerState 'tcp://127.0.0.1:60081', name: 2, status: running, memory: 0, processing: 0>
2026-04-02 16:25:24,650 - distributed.scheduler - WARNING - Worker failed to heartbeat for 309s; attempting restart: <WorkerState 'tcp://127.0.0.1:60082', name: 1, status: running, memory: 0, processing: 0>
2026-04-02 16:25:25,361 - distributed.nanny - WARNING - Restarting worker
2026-04-02 16:25:25,404 - distributed.nanny - WARNING - Restarting worker
2026-04-02 16:25:25,413 

In [66]:
%%time
# Create individual references in memory:
open_vds_par = delayed(open_virtual_dataset)
tasks = [
    open_vds_par(p, indexes={}, reader_options=reader_opts, loadable_variables=coord_vars) 
    for p in data_links[:100] # All passes
    ]
virtual_ds_list = list(da.compute(*tasks)) # The xr.concat() function below needs a list rather than a tuple.
print(len((virtual_ds_list)))

100
CPU times: user 9.61 s, sys: 4.61 s, total: 14.2 s
Wall time: 2min 49s


In [67]:
# Create the combined reference. Extract out a start time for each granule, and the cycle and pass number. Add the netCDF filename too.
orbit_start = []
cycle_num = []
pass_num = []
file_list = []

for g in granule_info[:len(data_links[:])]:
    datetime_str = g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime']
    datetime_obj = np.datetime64(datetime_str)
    orbit_start.append(datetime_obj)
    track = g['umm']['SpatialExtent']['HorizontalSpatialDomain']['Track']
    cycle = track['Cycle']
    cycle_num.append(cycle)
    ppass = track['Passes'][0]['Pass']
    pass_num.append(ppass)
    data_links = g.data_links(access="https")[0]
    filename = Path(data_links).name
    file_list.append(filename)
    #print(datetime_obj, cycle, ppass, filename)


/var/folders/c4/bkn3x29559ggcdlgd9grtx4m0000gp/T/ipykernel_75113/3729456310.py:9: UserWarning: no explicit representation of timezones available for np.datetime64
  datetime_obj = np.datetime64(datetime_str)


In [69]:
# Zip all the variables together in preparation for sorting
records = list(zip(
    orbit_start,
    cycle_num,
    pass_num,
    file_list,
    virtual_ds_list
))

# Sort them all at once using orbit_start
records_sorted = sorted(
    records,
    key=lambda x: np.datetime64(x[0])
)

# More robust way to sort the orbits and all variables
#pairs = sorted(
 #   zip(orbit_start, virtual_ds_list),
#    key=lambda x: np.datetime64(x[0])   # ensures proper datetime sorting
#)

# Unpack everything
orbit_start_sorted, cycle_sorted, pass_sorted, file_list_sorted, virtual_ds_list_sorted = zip(*records_sorted)

orbit_start_sorted = np.array(orbit_start_sorted)
cycle_sorted = np.array(cycle_sorted)
pass_sorted = np.array(pass_sorted)
file_list_sorted = list(file_list_sorted)
virtual_ds_list_sorted = list(virtual_ds_list_sorted)

In [70]:
# Add the new concat dimension of "granule" that are sequential numbers
granule_index = np.arange(len(virtual_ds_list_sorted))

In [71]:
# Construct the VDS from the list
coords_list = ['latitude', 'longitude' ]
virtual_ds_combined = xr.concat(
    virtual_ds_list_sorted,
    dim="granule",
    coords=coords_list,
    compat='override',
    combine_attrs='drop_conflicts'   
)

virtual_ds_combined = virtual_ds_combined.assign_coords(
    granule=("granule", granule_index),
    orbit=("granule", orbit_start_sorted),
    cycle=("granule", cycle_sorted),
    ppass=("granule", pass_sorted),
    filename=("granule", file_list_sorted)
)

/var/folders/c4/bkn3x29559ggcdlgd9grtx4m0000gp/T/ipykernel_75113/1570592216.py:11: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  virtual_ds_combined = virtual_ds_combined.assign_coords(


In [72]:
virtual_ds_combined

<xarray.Dataset> Size: 9GB
Dimensions:                                (granule: 100, num_lines: 9866,
                                            num_pixels: 69, num_sides: 2)
Coordinates:
    latitude                               (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
    longitude                              (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
  * granule                                (granule) int64 800B 0 1 2 ... 98 99
    orbit                                  (granule) datetime64[ns] 800B 2023...
    cycle                                  (granule) int64 800B 1 1 1 ... 1 1 1
    ppass                                  (granule) int64 800B 134 135 ... 234
    filename                               (granule) <U71 28kB 'SWOT_L2_LR_SS...
Dimensions without coordinates: num_lines, num_pixels, num_sides
Data variables: (12/24)
    time                                   (granule, num_lines) float64 8MB M...
    time_tai                               (granule, num_lines) float64 8MB M...
    ssh_karin                              (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
    ssh_karin_qual                         (granule, num_lines, num_pixels) uint32 272MB ManifestArray<shape=(100, 9866, 69), dtype=uint32, chunks=(1...
    ssh_karin_uncert                       (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
    ssha_karin                             (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
    ...                                     ...
    mean_sea_surface_cnescls               (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
    mean_sea_surface_cnescls_uncert        (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
    geoid                                  (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
    internal_tide_hret                     (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
    height_cor_xover                       (granule, num_lines, num_pixels) float64 545MB ManifestArray<shape=(100, 9866, 69), dtype=float64, chunks=(...
    height_cor_xover_qual                  (granule, num_lines, num_pixels) uint8 68MB ManifestArray<shape=(100, 9866, 69), dtype=uint8, chunks=(1, ...
Attributes: (12/26)
    Conventions:                      CF-1.7
    title:                            Level 2 Low Rate Sea Surface Height Dat...
    institution:                      CNES
    source:                           Ka-band radar interferometer
    platform:                         SWOT
    reference_document:               D-56407_SWOT_Product_Description_L2_LR_SSH
    ...                               ...
    xref_reforbittrack_files:         SWOT_RefOrbitTrack125mPass1_Nom_2000010...
    xref_pole_location_file:          SMM_PO1_AXXCNE20250411_020000_19900101_...
    xref_geco_database_version:       v107
    ellipsoid_semi_major_axis:        6378137.0
    ellipsoid_flattening:             0.0033528106647474805
    references:                       V1.4.1

In [20]:
pd.Index(orbit_start).is_monotonic_increasing

True

In [21]:
pd.Index(orbit_start_sorted).is_monotonic_increasing

True

In [22]:
# Save in JSON and PARQUET format:
fname_combined_json = 'ref_combined_swotl2_verD_full_https.json'
fname_combined_parq = 'rref_combined_swotl2_verD_full_https.parq'
virtual_ds_combined.virtualize.to_kerchunk(fname_combined_json, format='json')
virtual_ds_combined.virtualize.to_kerchunk(fname_combined_parq, format='parquet')

In [25]:
# shutdown the cluster
client.shutdown()

### 2.2.1  Using a Dask Management service (coiled) to parallize and  process the entire record

In [ ]:
%%time

## --------------------------------------------
## Create single reference files with parallel computing using Coiled
## --------------------------------------------

# Wrap `open_virtual_dataset()` into coiled function and copy to mulitple VM's:
open_vds_par = coiled.function(
    region="us-west-2", spot_policy="on-demand", 
    vm_type="m6i.large", n_workers=15
    )(open_virtual_dataset)

# Begin computations for the enire record:
results = open_vds_par.map(
    data_links[:], indexes={}, 
    reader_options=reader_opts, loadable_variables=coord_vars
    )

virtual_ds_list = []
for r in results:
    virtual_ds_list.append(r)

In [13]:
# Create the combined reference. Extract out a start time for each granule, and the cycle and pass number. Add the netCDF filename too.
orbit_start = []
cycle_num = []
pass_num = []
file_list = []

for g in granule_info[:len(data_links[:])]:
    datetime_str = g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime']
    datetime_obj = np.datetime64(datetime_str)
    orbit_start.append(datetime_obj)
    track = g['umm']['SpatialExtent']['HorizontalSpatialDomain']['Track']
    cycle = track['Cycle']
    cycle_num.append(cycle)
    ppass = track['Passes'][0]['Pass']
    pass_num.append(ppass)
    data_links = g.data_links(access="https")[0]
    filename = Path(data_links).name
    file_list.append(filename)
    #print(datetime_obj, cycle, ppass, filename)

/var/folders/c4/bkn3x29559ggcdlgd9grtx4m0000gp/T/ipykernel_75113/3512837864.py:8: UserWarning: no explicit representation of timezones available for np.datetime64
  datetime_obj = np.datetime64(datetime_str)


In [14]:
# Zip all the variables together in preparation for sorting
records = list(zip(
    orbit_start,
    cycle_num,
    pass_num,
    file_list,
    virtual_ds_list
))

# Sort them all at once using orbit_start
records_sorted = sorted(
    records,
    key=lambda x: np.datetime64(x[0])
)

# More robust way to sort the orbits and all variables
#pairs = sorted(
 #   zip(orbit_start, virtual_ds_list),
#    key=lambda x: np.datetime64(x[0])   # ensures proper datetime sorting
#)

# Unpack everything
orbit_start_sorted, cycle_sorted, pass_sorted, file_list_sorted, virtual_ds_list_sorted = zip(*records_sorted)

orbit_start_sorted = np.array(orbit_start_sorted)
cycle_sorted = np.array(cycle_sorted)
pass_sorted = np.array(pass_sorted)
file_list_sorted = list(file_list_sorted)
virtual_ds_list_sorted = list(virtual_ds_list_sorted)

In [ ]:
# Add the new concat dimension of "granule" that are sequential numbers
granule_index = np.arange(len(virtual_ds_list_sorted))

In [16]:
# Construct the VDS from the list
coords_list = ['latitude', 'longitude' ]
virtual_ds_combined = xr.concat(
    virtual_ds_list_sorted,
    dim="granule",
    coords=coords_list,
    compat='override',
    combine_attrs='drop_conflicts'   
)

virtual_ds_combined = virtual_ds_combined.assign_coords(
    granule=("granule", granule_index),
    orbit=("granule", orbit_start_sorted),
    cycle=("granule", cycle_sorted),
    ppass=("granule", pass_sorted),
    filename=("granule", file_list_sorted)
)

/var/folders/c4/bkn3x29559ggcdlgd9grtx4m0000gp/T/ipykernel_75113/4255951704.py:11: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  virtual_ds_combined = virtual_ds_combined.assign_coords(


In [ ]:
# Save in JSON and PARQUET format:
fname_combined_json = 'ref_combined_swotl2_D_https_sorted.json'
fname_combined_parq = 'ref_combined_swotl2_D_https_sorted.parq'
virtual_ds_combined.virtualize.to_kerchunk(fname_combined_json, format='json')
virtual_ds_combined.virtualize.to_kerchunk(fname_combined_parq, format='parquet')

In [ ]:
virtual_ds_combined

In [ ]:
pd.Index(orbit_start).is_monotonic_increasing

In [ ]:
pd.Index(orbit_start_sorted).is_monotonic_increasing

In [ ]:
# shutdown cluster
open_vds_par.cluster.shutdown()